In [1]:
# Test hipotezy: rozbieznosc wyniku Moskwy miedzy 20260716a.ipynb (sigma=3.12,
# dane mosc_data.csv) a 20260717b.ipynb (sigma_mc=2.484 dla wiersza MOSC/HRMS,
# dane z csv_data_allstations/*_60min.csv) NIE wynika z liczby symulacji MC ani
# z dtype/units (sprawdzone statycznie - oba pobrania uzywaja
# dtype=corr_for_efficiency, units=0 - patrz Pobieranie_Oulu_+_Mosc.ipynb cell 10
# vs download_allstations_history.py).
#
# Hipoteza do sprawdzenia: mosc_data.csv to natywny produkt 6h z NMDB
# (poproszony wprost o tresolution=360/120 wg Pobieranie_Oulu_+_Mosc.ipynb cell 9,
# potwierdzone: timestampy co 6h w pliku), natomiast dane z
# csv_data_allstations/*_60min.csv maja MIMO NAZWY realna rozdzielczosc ~2h
# (sprawdzone awk-iem: kolumna MOSC ma punkty co 2h, nie 60 min - serwer NMDB w
# trybie allstations sam ograniczal zwracana rozdzielczosc, patrz 20260717.txt
# pkt 5). Resampling tych 2h danych do 6h srednia z 3 rzadkich probek to inny
# (bardziej zaszumiony) estymator niz natywny agregat 6h liczony przez NMDB z
# pelnej rozdzielczosci surowych zliczen - to podejrzenie o mechanizm
# rozbieznosci, sprawdzane ponizej wprost na liczbach.
import glob
import numpy as np
import pandas as pd
from scipy.stats import binom, norm

MOSC_PATH = "../data/mosc_data.csv"
ALLSTATIONS_GLOB = "../data/csv_data_allstations/*_60min.csv"
USGS_PATH = "../data/usgs_data/usgs_m4_2005_2025.csv"

def load_moscow_native6h():
    df = pd.read_csv(MOSC_PATH, parse_dates=["datetime"])
    return df.set_index("datetime").sort_index()["value"]

def load_moscow_from_allstations():
    files = sorted(glob.glob(ALLSTATIONS_GLOB))
    dfs = []
    for f in files:
        df = pd.read_csv(f, sep=";", na_values=["null"], usecols=lambda c: c in ("timestamp", "MOSC"))
        df["timestamp"] = pd.to_datetime(df["timestamp"])
        dfs.append(df)
    full = pd.concat(dfs, ignore_index=True).drop_duplicates(subset="timestamp").sort_values("timestamp")
    full = full.set_index("timestamp")
    mosc_native = pd.to_numeric(full["MOSC"], errors="coerce")
    mosc_native = mosc_native[mosc_native > 0].dropna()
    return mosc_native

def load_earthquakes(min_mag=4.0):
    df = pd.read_csv(USGS_PATH, usecols=["time", "mag"])
    df["time"] = pd.to_datetime(df["time"], utc=True).dt.tz_localize(None)
    df = df[df["mag"] >= min_mag]
    return df.set_index("time")["mag"].sort_index()

cr_native6h = load_moscow_native6h()
cr_allst_native = load_moscow_from_allstations()
eq = load_earthquakes(min_mag=4.0)

print(f"mosc_data.csv (natywne, wg CLAUDE.md 6h): {len(cr_native6h)} pomiarow, "
      f"{cr_native6h.index.min()} .. {cr_native6h.index.max()}")
print(f"allstations MOSC (natywne, przed resamplingiem): {len(cr_allst_native)} pomiarow, "
      f"{cr_allst_native.index.min()} .. {cr_allst_native.index.max()}")

# Sprawdzenie realnej rozdzielczosci obu zrodel (mediana odstepu miedzy
# kolejnymi pomiarami) - to bezposrednio testuje hipoteze "~2h mimo nazwy
# 60min" vs natywne 6h.
delta_native6h = cr_native6h.index.to_series().diff().dropna()
delta_allst = cr_allst_native.index.to_series().diff().dropna()
print(f"\nmediana odstepu mosc_data.csv: {delta_native6h.median()}")
print(f"mediana odstepu allstations MOSC (przed resamplingiem): {delta_allst.median()}")

cr_allst_6h = cr_allst_native.resample("6h").mean()
print(f"\nallstations MOSC po .resample('6h').mean(): {len(cr_allst_6h)} pomiarow")

mosc_data.csv (natywne, wg CLAUDE.md 6h): 91824 pomiarow, 1960-01-01 00:00:00 .. 2025-03-23 18:00:00
allstations MOSC (natywne, przed resamplingiem): 39749 pomiarow, 2011-01-01 00:00:00 .. 2019-12-26 22:00:00

mediana odstepu mosc_data.csv: 0 days 06:00:00
mediana odstepu allstations MOSC (przed resamplingiem): 0 days 02:00:00

allstations MOSC po .resample('6h').mean(): 13128 pomiarow


In [2]:
# Ten sam test co w 20260716a.ipynb/20260717b.ipynb (kopia funkcji dla
# spojnosci - nie importujemy z innego notebooka, zeby ten byl samodzielny).
def cosmoseismic_stat(cr, eq, t0, P_days, d_days, m, dt_days, return_votes=False):
    N = int(P_days // d_days)
    edges = pd.date_range(t0, periods=N + 1, freq=pd.Timedelta(days=d_days))
    eq_edges = edges + pd.Timedelta(days=dt_days)

    cr_cats = pd.cut(cr.index, edges, right=False)
    cr_binned = cr.groupby(cr_cats, observed=False).mean().reindex(cr_cats.categories)
    cr_vals = cr_binned.to_numpy()

    eq_in_range = eq[(eq.index >= eq_edges[0]) & (eq.index < eq_edges[-1])]
    eq_cats = pd.cut(eq_in_range.index, eq_edges, right=False)
    eq_binned = eq_in_range.groupby(eq_cats, observed=False).sum().reindex(eq_cats.categories, fill_value=0.0)
    sm_vals = eq_binned.to_numpy()

    nCR_i, nCR_im1 = cr_vals[1:], cr_vals[:-1]
    dCR = nCR_i - nCR_im1
    Sm = sm_vals[1:]

    med_Sm = np.nanmedian(Sm)
    med_dCR = np.nanmedian(np.abs(dCR))

    A = Sm / med_Sm - 1
    B = np.abs(dCR) / med_dCR - 1

    valid = (
        (A != 0) & (B != 0) &
        (nCR_i > 0) & (nCR_im1 > 0) &
        (Sm > 0) &
        ~np.isnan(A) & ~np.isnan(B)
    )

    c_valid = (A * B)[valid]
    Np, Nm = int((c_valid > 0).sum()), int((c_valid < 0).sum())
    n_total = Np + Nm

    result = dict(N=N, N_valid=n_total, Np=Np, Nm=Nm)
    if n_total == 0:
        result.update(PPDF=np.nan, PCDF=np.nan, sigma=np.nan)
    else:
        ppdf = binom.pmf(Np, n_total, 0.5)
        pcdf = binom.sf(Np - 1, n_total, 0.5)
        result.update(PPDF=ppdf, PCDF=pcdf, sigma=norm.isf(pcdf))

    if return_votes:
        votes = np.full(len(dCR), np.nan)
        votes[valid] = np.sign(c_valid)
        result["votes"] = votes
        result["bin_edges"] = edges[1:-1]  # start kazdego binu i (od 2. do N-1)

    return result


def full_d_scan(cr, eq, t0, P_days, m, dt_days, d_range):
    return {d: cosmoseismic_stat(cr, eq, t0, P_days, d, m, dt_days)["PCDF"] for d in d_range}


t0 = pd.Timestamp("2013-11-14 07:00:00")
P_days, m, dt_days = 1675, 4.0, 15

In [3]:
# Deterministyczny skan d=1..30 (bez Monte Carlo - to szybkie, sekundy, nie
# wymaga czekania) - porownanie best_pcdf/best_d obu zrodel danych Moskwy przy
# IDENTYCZNYM t0/P_days/m/dt_days. Referencje z poprzednich sesji:
#   20260716.txt: mosc_data.csv -> best_pcdf=1.162e-05 przy d=5, sigma_mc=3.12 (n=1e5)
#   20260717.txt: allstations MOSC -> best_pcdf=7.587e-05 przy d=5, sigma_mc=2.484 (n=2000)
sim_native6h = full_d_scan(cr_native6h, eq, t0, P_days, m, dt_days, range(1, 31))
sim_allst6h = full_d_scan(cr_allst_6h, eq, t0, P_days, m, dt_days, range(1, 31))

best_d_native = min(sim_native6h, key=sim_native6h.get)
best_d_allst = min(sim_allst6h, key=sim_allst6h.get)

print(f"mosc_data.csv (natywne 6h):        best_d={best_d_native}, best_pcdf={sim_native6h[best_d_native]:.3e}")
print(f"allstations MOSC (2h->6h resample): best_d={best_d_allst}, best_pcdf={sim_allst6h[best_d_allst]:.3e}")
print(f"stosunek best_pcdf (allstations / natywne): {sim_allst6h[best_d_allst] / sim_native6h[best_d_native]:.2f}x")

print("\nPelny skan PCDF(d) obu zrodel obok siebie:")
scan_cmp = pd.DataFrame({
    "PCDF_native6h": pd.Series(sim_native6h),
    "PCDF_allst6h": pd.Series(sim_allst6h),
})
scan_cmp["ratio"] = scan_cmp["PCDF_allst6h"] / scan_cmp["PCDF_native6h"]
pd.set_option("display.float_format", lambda x: f"{x:.4g}")
print(scan_cmp)

mosc_data.csv (natywne 6h):        best_d=5, best_pcdf=1.162e-05
allstations MOSC (2h->6h resample): best_d=5, best_pcdf=7.587e-05
stosunek best_pcdf (allstations / natywne): 6.53x

Pelny skan PCDF(d) obu zrodel obok siebie:
    PCDF_native6h  PCDF_allst6h  ratio
1          0.3387        0.3037 0.8966
2         0.06846       0.03885 0.5674
3       0.0006176     0.0008295  1.343
4         0.02473       0.07038  2.846
5       1.162e-05     7.587e-05  6.528
6          0.0409        0.1039   2.54
7         0.08666       0.02995 0.3457
8          0.4176        0.2229 0.5336
9          0.8125        0.8125      1
10          0.349        0.4691  1.344
11         0.2562        0.1628 0.6355
12         0.2219        0.1342 0.6049
13         0.8144        0.6397 0.7854
14        0.02637        0.1156  4.384
15         0.8529        0.8529      1
16            0.5        0.8838  1.768
17            0.5        0.6591  1.318
18         0.6988        0.8259  1.182
19         0.7423        0.6677 0.

In [4]:
# Mechanizm: dla d=5 (best_d wg obu zrodel) - ile pojedynczych "glosow" testu
# znaku (znak A*B w kazdym binie) roznie sie miedzy dwoma zrodlami danych CR,
# oraz jak bardzo roznia sie same wartosci CR bin-po-binie (po zresamplingu
# do 6h obu zrodel na wspolna siatke czasowa). To pokazuje wprost, czy
# rozbieznosc sigma bierze sie z drobnych przesuniec kilku granicznych glosow
# (jak podejrzewano w 20260717.txt dla oryginalnego zbioru vs allstations przy
# t0 Augera), czy z czegos wiekszego.
d_check = 5
r_native = cosmoseismic_stat(cr_native6h, eq, t0, P_days, d_check, m, dt_days, return_votes=True)
r_allst = cosmoseismic_stat(cr_allst_6h, eq, t0, P_days, d_check, m, dt_days, return_votes=True)

print(f"mosc_data.csv (d={d_check}):        Np={r_native['Np']}, Nm={r_native['Nm']}, PCDF={r_native['PCDF']:.3e}")
print(f"allstations MOSC (d={d_check}):     Np={r_allst['Np']}, Nm={r_allst['Nm']}, PCDF={r_allst['PCDF']:.3e}")

votes_native = pd.Series(r_native["votes"], index=r_native["bin_edges"])
votes_allst = pd.Series(r_allst["votes"], index=r_allst["bin_edges"])
both_valid = votes_native.notna() & votes_allst.notna()
flips = (votes_native[both_valid] != votes_allst[both_valid]).sum()
print(f"\nBinow z waznym glosem w obu zrodlach: {both_valid.sum()}")
print(f"Binow gdzie znak glosu (A*B) sie ROZNI miedzy zrodlami: {flips} "
      f"({100 * flips / both_valid.sum():.1f}%)")

# Porownanie samych wartosci CR bin-po-binie (wspolna siatka czasowa, po
# zresamplingu obu do 6h) - czy to male szumowe roznice, czy systematyczne
# przesuniecie/rozjazd.
common_idx = cr_native6h.index.intersection(cr_allst_6h.index)
common_idx = common_idx[(common_idx >= t0) & (common_idx <= t0 + pd.Timedelta(days=P_days))]
diff = (cr_native6h.loc[common_idx] - cr_allst_6h.loc[common_idx])
rel_diff = diff / cr_native6h.loc[common_idx]
print(f"\nWspolnych binow czasowych (6h) w oknie testu: {len(common_idx)}")
print(f"Roznica wartosci CR (native - allstations): mediana={diff.median():.4f}, "
      f"odchylenie std={diff.std():.4f}")
print(f"Roznica wzgledna: mediana={rel_diff.median()*100:.3f}%, "
      f"odchylenie std={rel_diff.std()*100:.3f}%, max={rel_diff.abs().max()*100:.2f}%")
print(f"Korelacja Pearsona (wspolna siatka): {cr_native6h.loc[common_idx].corr(cr_allst_6h.loc[common_idx]):.5f}")

mosc_data.csv (d=5):        Np=206, Nm=128, PCDF=1.162e-05
allstations MOSC (d=5):     Np=202, Nm=132, PCDF=7.587e-05

Binow z waznym glosem w obu zrodlach: 334
Binow gdzie znak glosu (A*B) sie ROZNI miedzy zrodlami: 12 (3.6%)

Wspolnych binow czasowych (6h) w oknie testu: 6650
Roznica wartosci CR (native - allstations): mediana=0.0000, odchylenie std=3.2076
Roznica wzgledna: mediana=0.000%, odchylenie std=1.661%, max=75.00%
Korelacja Pearsona (wspolna siatka): 0.97639
